In [39]:
from langchain_community.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
import requests
import os
from dotenv import load_dotenv
from langchain_core.tools import InjectedToolArg
from typing import Annotated
load_dotenv()

True

In [40]:
### Conversionn Rate Tool



@tool(parse_docstring=True)
def get_conversion_rate(base_currency: str, target_currency: str) -> float:
    """Fetch the current exchange rate between two currencies.

    Use this tool whenever you need the live exchange rate from one currency to
    another, either because the user asked for the rate directly, or because you
    need a rate before converting an amount. Never guess or recall
    exchange rates from memory, because they change constantly.

    Args:
        base_currency: The 3-letter ISO 4217 code of the currency being converted FROM, in uppercase, e.g. "USD", "EUR", "PKR".
        target_currency: The 3-letter ISO 4217 code of the currency being converted TO, in uppercase, e.g. "GBP", "JPY", "PKR".

    Returns:
        The Conversion rate between two currencies
    """
    api_key = os.getenv("EXCHANGE_API_KEY")
    url = f"https://v6.exchangerate-api.com/v6/{api_key}/pair/{base_currency.upper()}/{target_currency.upper()}"

    response = requests.get(url)
    data = response.json()

    if data.get("result") != "success":
        raise ValueError(f"API error: {data.get('error-type', data)}")

    return data["conversion_rate"]





### Currency Convertor Tool
@tool(parse_docstring=True)
def convert_currency(base_currency_value: float,   conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """Convert an amount of money from a base currency to a target currency using a known conversion rate.

    Use this tool whenever the user wants to know how much a given amount in one
    currency is worth in another currency. You MUST already have the conversion
    rate before calling this tool.Do not guess or invent the rate, and do not do the
    multiplication yourself.

    Args:
        base_currency_value: The amount of money in the base (source) currency, e.g. 250.0 for 250 EUR.
        conversion_rate: The exchange rate for 1 unit of the base currency expressed in the target currency, e.g. 0.865 for EUR to GBP. Obtain it from `get_conversion_factor`.

    Returns:
        The converted amount in the target currency as a float.
    """
    return base_currency_value * conversion_rate

convert_currency.invoke({"base_currency_value":1,"conversion_rate":get_conversion_rate.invoke({"base_currency":"USD","target_currency":"PKR"})})

277.0451

In [41]:
### Tools Attaching to LLM

tools={"get_conversion_rate":get_conversion_rate
       ,"convert_currency":convert_currency}

primary_llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.7,
 
)

backup_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",   
    temperature=0.7,
)

llm_with_tools = primary_llm.bind_tools([get_conversion_rate,convert_currency]).with_fallbacks(
    [backup_llm.bind_tools([get_conversion_rate,convert_currency])]
)

In [42]:
### Creating History List
messages=[HumanMessage("Convert the 56 USD to PkR")]

In [43]:
llm_with_tools.invoke(messages)

AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_conversion_rate', 'arguments': '{"base_currency": "USD", "target_currency": "PKR"}'}, '__gemini_function_call_thought_signatures__': {'b7caf5e0-2f86-4427-8730-5c46ac78d203': 'CvUhAWkUfRP3DUK0N8gRz+EdvQnY2Z/nlnaBRvph4M296Ek6udC1emfkgmh6fcE5lQ/6qjS6MnxSnqNEV1nrzms3DDXLFXrV3qAKV6IQKzKfmmnvWHgTWBZ3WpNXEsuzrw8MvHcp4SiFhfzVfLxcpZdtvUZ3a07VX8gOKQP1qZrS997gHhRTRc6hGqYy5n2FQ67GZdpKv1Hthx8y45ygXBdYflyFfnZ8PW2KfboPaYSjsaSZAvVVhY897pHTwop9BmPE2jcy5+hHVPLOdoKaRhXNi9e9pFMVVn+wwfsFUtkfujW5IGQqcDH3nJxD6lSJAIMWwdEDh3DJ2oIUszZ3GPAVbKbCkUwEck8764Ve6shlNHwxqXzTq7yg+Ym7oPEWoIoA3fOV8X9qjIJtQDJJ2cNaAPZI9E+/ZhGSOvJuLa2ZO4S1m9owKp/Z3MQVilx05WPmbd85hwVyKAncA8c6BQ5qB1cGqfEryl2ebJaU8xcC3yJM+NY9p0SPsfHMKHEOvujp1alRJSP/EXZO9j+7TdbkLS+w5/RkThX7MVelIz/M9S/RtKqST4y2fXhwaB2Plq5atbRAc9EkFPVsDLVjqXxTML/CxMv19gqSrjUVAtTXTZHbLLxX4izUwYRCWxv/UHrQ8QLyHzFm4L5cTygQk1YG0+/VyXn3fTJ367AU6g9JWv1uwDghIo0CiD91QsOZNWDbDx/4SckOuFbPaHBJfCTx4Dco+t9sWi0jxQaVIPhLEG+IF